# Hallmark Demo: add, commit, and checkout

This notebook walks through the basic hallmark workflow:

1. Initialize a hallmark repo
2. Create data files, `add` them, and `commit`
3. `checkout` a new branch
4. Add different files on the new branch and commit
5. `checkout` back to `main` — files swap in place
6. Try to checkout with uncommitted changes — hallmark refuses

## Setup

In [1]:
# auto-reload modules when they change
%load_ext autoreload
%autoreload 2

In [2]:
%%bash
rm -rf repo1
mkdir repo1

## 1. Initialize the hallmark repo

This creates a `repo1/.hm/` directory. The `.hm/` is a git repo that tracks
`data.tsv` (the file manifest), `config.yml`, and `meta.yml`.
The shared `objects/` store is used for file bytes, but it is created lazily on
the first `commit()` that actually stores blobs.


In [3]:
from hallmark import Repo

repo = Repo.init("repo1")

In [4]:
%%bash
echo "=== .hm directory ==="
ls repo1/.hm/
echo ""
echo "=== objects stored so far (0 before first commit) ==="
find repo1/.hm/objects -type f 2>/dev/null | wc -l


=== .hm directory ===
config.yml
data.tsv
meta.yml
README.md

=== objects stored so far (0 before first commit) ===
       0


## 2. Create data files on `main`

We create 10 `.h5` files with different spin and inclination values.
These are just dummy files for the demo — in a real project, these would be
simulation outputs or observational data.

In [5]:
%%bash
cd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do
    echo "$f" > "$f"
done
echo "Files in repo1/:"
ls *.h5

Files in repo1/:
a0_i0.h5
a0_i120.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5


## 3. Add files and commit on `main`

`repo.add()` does two things:
- Parses the format string to find matching files and extract parameters (spin, inc)
- Computes a sha1 checksum for each file and stages a manifest containing only `sha1` and `path` in `data.tsv`

`repo.commit()` does two things:
- Copies each file's bytes into the shared `objects/` store, keyed by sha1
- Commits `data.tsv` in the `.hm/` git repo

After commit, the file bytes are safe in `objects/`. Even if the files get
deleted from the working directory, they can be restored.


In [6]:
pf = repo.add("a{spin}_i{inc}.h5")
print(pf)

            path  spin    inc
0    a0.75_i0.h5  0.75    0.0
1  a0.75_i120.h5  0.75  120.0
2   a0.75_i30.h5  0.75   30.0
3   a0.75_i60.h5  0.75   60.0
4   a0.75_i90.h5  0.75   90.0
5       a0_i0.h5  0.00    0.0
6     a0_i120.h5  0.00  120.0
7      a0_i30.h5  0.00   30.0
8      a0_i60.h5  0.00   60.0
9      a0_i90.h5  0.00   90.0


In [7]:
repo.commit("add spin and inclination files")

True

In [8]:
%%bash
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l

Objects stored after commit:
      10


10 files added, 10 objects stored. Each object is a copy of a file's bytes,
named by its sha1 hash (split as `sha1[:2]/sha1[2:]`, same as git).

## 4. Checkout a new branch

This works like `git checkout -b experiment`. The `.hm/` repo switches to a new
branch called `experiment`. Since it just branched off `main`, the files are
the same — `data.tsv` is identical, and the same files are restored from `objects/`.

What happens under the hood:
1. Validate that tracked files in the working directory still match the current branch's `data.tsv`
2. Delete current tracked files from the working directory
3. Run `git checkout -b experiment` inside `.hm/`
4. Read the new branch's `data.tsv`
5. Restore each tracked file from `objects/` using hardlinks

Untracked files are left alone unless they block restoration of a tracked path.


In [9]:
repo.checkout("experiment")

True

In [10]:
%%bash
echo "Files in repo1/ after checkout:"
ls repo1/*.h5
echo ""
echo "Current .hm branch:"
git -C repo1/.hm branch --show-current

Files in repo1/ after checkout:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5

Current .hm branch:
experiment


Same files as before — because `experiment` just branched off `main`.

## 5. Add new files on `experiment` and commit

Now we create different files on the `experiment` branch and track them.
After commit, `objects/` grows — it now holds files from both branches, while
`data.tsv` on each branch still lists only that branch's manifest.


In [11]:
%%bash
cd repo1
for f in b{0.5,1.0}_i{0,45,90,135,155}.h5; do
    echo "$f" > "$f"
done
echo "All files in repo1/ now:"
ls *.h5

All files in repo1/ now:
a0_i0.h5
a0_i120.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5
b0.5_i0.h5
b0.5_i135.h5
b0.5_i155.h5
b0.5_i45.h5
b0.5_i90.h5
b1.0_i0.h5
b1.0_i135.h5
b1.0_i155.h5
b1.0_i45.h5
b1.0_i90.h5


In [12]:
pf_exp = repo.add("b{spin}_i{inc}.h5")
print(pf_exp)

           path  spin    inc
0    b0.5_i0.h5   0.5    0.0
1  b0.5_i135.h5   0.5  135.0
2  b0.5_i155.h5   0.5  155.0
3   b0.5_i45.h5   0.5   45.0
4   b0.5_i90.h5   0.5   90.0
5    b1.0_i0.h5   1.0    0.0
6  b1.0_i135.h5   1.0  135.0
7  b1.0_i155.h5   1.0  155.0
8   b1.0_i45.h5   1.0   45.0
9   b1.0_i90.h5   1.0   90.0


In [13]:
repo.commit("add experiment files")

True

In [14]:
%%bash
echo "Objects stored after experiment commit:"
find repo1/.hm/objects -type f | wc -l

Objects stored after experiment commit:
      20


20 objects total — 10 from `main` + 10 new from `experiment`.
The objects store holds everything across all branches, while each branch keeps
its own committed `data.tsv`.


## 6. Checkout back to `main`

This is where the magic happens. The working directory gets rewritten in place:
1. All of experiment's tracked files get deleted from the working directory
2. `.hm/` switches to the `main` branch
3. Main's files get restored from `objects/`

No data is lost — experiment's file bytes are still in `objects/`, but `main`
is rebuilt only from `main`'s `data.tsv`.


In [15]:
repo.checkout("main")

True

In [16]:
%%bash
echo "Files in repo1/ after checkout main:"
ls repo1/*.h5
echo ""
echo "Current .hm branch:"
git -C repo1/.hm branch --show-current
echo ""
echo "Objects still in store (nothing lost):"
find repo1/.hm/objects -type f | wc -l

Files in repo1/ after checkout main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5

Current .hm branch:
main

Objects still in store (nothing lost):
      20


Only main's 10 `a*` files are in the working directory. The `b*` files are gone
from the directory but their bytes are safe in `objects/`. Checkout back to
`experiment` and they come right back.


## 7. Checkout back to `experiment` to verify

Let's go back to `experiment` to confirm the files are restored correctly.

In [17]:
repo.checkout("experiment")


True

In [18]:
%%bash
echo "Files in repo1/ after checkout experiment:"
ls repo1/*.h5
echo ""
echo "Current .hm branch:"
git -C repo1/.hm branch --show-current

Files in repo1/ after checkout experiment:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5
repo1/b0.5_i0.h5
repo1/b0.5_i135.h5
repo1/b0.5_i155.h5
repo1/b0.5_i45.h5
repo1/b0.5_i90.h5
repo1/b1.0_i0.h5
repo1/b1.0_i135.h5
repo1/b1.0_i155.h5
repo1/b1.0_i45.h5
repo1/b1.0_i90.h5

Current .hm branch:
experiment


All 20 files are back — the 10 from `main` (inherited when we branched) plus
the 10 we added on `experiment`.

## 8. Uncommitted changes block checkout

Just like git, hallmark will not let you checkout if tracked files have local
changes or if hallmark state has been added but not committed.


In [22]:
%%bash
cd repo1
echo "c0_i20.h5" > c0_i20.h5
echo "Files in repo1/ after creating c0_i20.h5:"
ls *.h5

In [23]:
repo.add("c{c}_i{i}.h5")

,path,c,i
0,c0_i20.h5,0.0,20.0


In [24]:
# this should fail because we have uncommitted changes
try:
    repo.checkout("main")
except RuntimeError as e:
    print(f"Error: {e}")

Error: you have uncommitted hallmark state changes — commit them before checkout


To fix this, either commit the changes first, or discard them.